# FIFA World Cup 2026 - Objective 2

- **Model 2.1:** predict the goal difference in a match (104 rows)
- **Model 2.2:** predict the goals scored by a team in a match (208 rows)


There are 8 explanatory variables for each dataset, and all of these are known prior to the match.

**Data sources**
- FBref: World Cup 2026 and 2022 Scores & Fixtures (exported as CSV)
- FBref match reports: 90 minute scores (if played extra time)
The latest update to FIFA's Men's World Ranking was released on 11 June 2026.

In [ ]:
pip install numpy pandas matplotlib seaborn scipy statsmodels scikit-learn jupyter ipykernel

In [ ]:
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from math import radians, sin, cos, asin, sqrt
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split, GroupShuffleSplit, KFold, GroupKFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Load the match results

In [ ]:
def get_score(score):
    score = re.sub(r"\(\d+\)", "", score).strip()
    home, away = score.split("–")
    return int(home), int(away)

def load_fixtures(fixtures_file, extra_time_file):
    df = pd.read_csv(fixtures_file)
    df = df.dropna(subset=["Score"]).reset_index(drop=True)

    df["home"] = df["Home"].apply(lambda x: x.rsplit(" ", 1)[0])
    df["away"] = df["Away"].apply(lambda x: x.split(" ", 1)[1])
    df["home_goals"] = df["Score"].apply(lambda x: get_score(x)[0])
    df["away_goals"] = df["Score"].apply(lambda x: get_score(x)[1])
    df["venue"] = df["Venue"].str.replace(" (Neutral Site)", "", regex=False)
    df["date"] = pd.to_datetime(df["Date"])
    df["knockout"] = (df["Round"] != "Group stage").astype(int)
    df["match_id"] = range(1, len(df) + 1)

    et = pd.read_csv(extra_time_file)
    for i, row in et.iterrows():
        match = (df["Date"] == row["Date"]) & (df["home"] == row["Home"])
        df.loc[match, "home_goals"] = row["Home goals at 90 min"]
        df.loc[match, "away_goals"] = row["Away goals at 90 min"]

    return df

matches = load_fixtures("data/fbref_fixtures_2026.csv", "data/fbref_extra_time_90min.csv")
matches_2022 = load_fixtures("data/fbref_fixtures_2022.csv", "data/fbref_extra_time_90min_2022.csv")

print("2026 matches:", len(matches))
print("2022 matches:", len(matches_2022))
matches[["date", "Round", "home", "home_goals", "away_goals", "away", "venue"]].head()

## 2. Load the FIFA ranking

In [ ]:
with open("data/fifa_ranking_2026-06-11_raw.json", encoding="utf-8") as f:
    ranking = json.load(f)["rankings"]

fifa_points = {}
for r in ranking:
    fifa_points[r["rankingItem"]["name"]] = r["rankingItem"]["totalPoints"]

fifa_points["Bosnia–Herz"] = fifa_points["Bosnia and Herzegovina"]

all_teams = set(matches["home"]) | set(matches["away"])
print("Teams:", len(all_teams))
print("Teams missing a ranking:", [t for t in all_teams if t not in fifa_points])

## 3. Stadium information

In [ ]:
# latitude, longitude, altitude (m), country
stadiums = {
    "Estadio Banorte": (19.3029, -99.1505, 2240, "Mexico"),
    "Estadio Akron": (20.6817, -103.4627, 1566, "Mexico"),
    "Estadio BBVA": (25.6690, -100.2445, 540, "Mexico"),
    "BMO Field": (43.6332, -79.4186, 76, "Canada"),
    "BC Place Stadium": (49.2768, -123.1120, 5, "Canada"),
    "AT&T Stadium": (32.7473, -97.0945, 184, "USA"),
    "Mercedes-Benz Stadium": (33.7554, -84.4008, 320, "USA"),
    "SoFi Stadium": (33.9535, -118.3392, 30, "USA"),
    "MetLife Stadium": (40.8135, -74.0745, 3, "USA"),
    "Gillette Stadium": (42.0909, -71.2643, 85, "USA"),
    "Hard Rock Stadium": (25.9580, -80.2389, 3, "USA"),
    "Reliant Stadium": (29.6847, -95.4107, 15, "USA"),
    "Levi's Stadium": (37.4030, -121.9700, 9, "USA"),
    "Lumen Field": (47.5952, -122.3316, 5, "USA"),
    "Lincoln Financial Field": (39.9008, -75.1675, 12, "USA"),
    "GEHA Field at Arrowhead Stadium": (39.0489, -94.4839, 270, "USA"),
}

def distance_km(venue1, venue2):
    lat1, lon1 = radians(stadiums[venue1][0]), radians(stadiums[venue1][1])
    lat2, lon2 = radians(stadiums[venue2][0]), radians(stadiums[venue2][1])
    a = sin((lat2 - lat1) / 2) ** 2 + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * asin(sqrt(a))

print("Venues without stadium info:", [v for v in matches["venue"].unique() if v not in stadiums])

## 4. One row per team per match

In [ ]:
def make_team_rows(df):
    rows = []
    for i, m in df.iterrows():
        rows.append({"match_id": m["match_id"], "date": m["date"], "venue": m["venue"],
                     "knockout": m["knockout"], "team": m["home"], "opponent": m["away"],
                     "goals_for": m["home_goals"], "goals_against": m["away_goals"], "is_home_team": 1})
        rows.append({"match_id": m["match_id"], "date": m["date"], "venue": m["venue"],
                     "knockout": m["knockout"], "team": m["away"], "opponent": m["home"],
                     "goals_for": m["away_goals"], "goals_against": m["home_goals"], "is_home_team": 0})
    teams = pd.DataFrame(rows)
    teams["points"] = np.where(teams["goals_for"] > teams["goals_against"], 3,
                      np.where(teams["goals_for"] == teams["goals_against"], 1, 0))
    teams = teams.sort_values(["team", "date"]).reset_index(drop=True)
    return teams

teams = make_team_rows(matches)
teams_2022 = make_team_rows(matches_2022)
print("Team-match rows:", len(teams))

## 5. Create the pre-match variables

Form is based on games played up until the previous match. The first match of a team is a match that a team has played first.
So we begin at the 2022 average for World Cup (as one additional game).

In [ ]:
avg_goals_2022 = teams_2022["goals_for"].mean()
avg_points_2022 = teams_2022["points"].mean()
print("2022 average goals per team per match:", round(avg_goals_2022, 3))
print("2022 average points per team per match:", round(avg_points_2022, 3))

g = teams.groupby("team")
games_before = g.cumcount()
goals_for_before = g["goals_for"].cumsum() - teams["goals_for"]
goals_against_before = g["goals_against"].cumsum() - teams["goals_against"]
points_before = g["points"].cumsum() - teams["points"]

teams["gf_form"] = (goals_for_before + avg_goals_2022) / (games_before + 1)
teams["ga_form"] = (goals_against_before + avg_goals_2022) / (games_before + 1)
teams["points_form"] = (points_before + avg_points_2022) / (games_before + 1)

teams["fifa_points"] = teams["team"].map(fifa_points)
teams["rest_days"] = g["date"].diff().dt.days
teams["prev_venue"] = g["venue"].shift()

travel = []
for i, row in teams.iterrows():
    if pd.isna(row["prev_venue"]):
        travel.append(0)
    else:
        travel.append(distance_km(row["prev_venue"], row["venue"]))
teams["travel_km"] = travel

teams["altitude"] = teams["venue"].apply(lambda v: stadiums[v][2])
teams["venue_country"] = teams["venue"].apply(lambda v: stadiums[v][3])
teams["host_at_home"] = (teams["team"] == teams["venue_country"]).astype(int)

teams[["team", "date", "goals_for", "gf_form", "ga_form", "points_form", "rest_days", "travel_km"]].head(8)

Add the opponent's values to each row.

In [ ]:
opp = teams[["match_id", "team", "fifa_points", "gf_form", "ga_form", "points_form",
             "rest_days", "travel_km", "host_at_home"]].copy()
opp.columns = ["match_id", "opponent", "opp_fifa_points", "opp_gf_form", "opp_ga_form", "opp_points_form",
               "opp_rest_days", "opp_travel_km", "opp_host_at_home"]
teams = teams.merge(opp, on=["match_id", "opponent"])
teams["rest_diff"] = (teams["rest_days"] - teams["opp_rest_days"]).fillna(0)

## 6. Build the two datasets

| Model 2.1 variables | Model 2.2 variables |
|---|---|
| team1_fifa_points | team_fifa_points |
| team2_fifa_points | opp_fifa_points |
| gf_form_diff | team_gf_form |
| ga_form_diff | opp_ga_form |
| points_form_diff | rest_diff |
| rest_diff | host_at_home |
| travel_diff | knockout |
| host_diff | altitude |

For Model 2.1 the target is team1 goals minus team2 goals, so most variables are differences
between the two teams.

In [ ]:
home_rows = teams[teams["is_home_team"] == 1]

d1 = pd.DataFrame({
    "match_id": home_rows["match_id"],
    "team1": home_rows["team"],
    "team2": home_rows["opponent"],
    "goal_diff": home_rows["goals_for"] - home_rows["goals_against"],
    "team1_fifa_points": home_rows["fifa_points"],
    "team2_fifa_points": home_rows["opp_fifa_points"],
    "gf_form_diff": home_rows["gf_form"] - home_rows["opp_gf_form"],
    "ga_form_diff": home_rows["ga_form"] - home_rows["opp_ga_form"],
    "points_form_diff": home_rows["points_form"] - home_rows["opp_points_form"],
    "rest_diff": home_rows["rest_diff"],
    "travel_diff": home_rows["travel_km"] - home_rows["opp_travel_km"],
    "host_diff": home_rows["host_at_home"] - home_rows["opp_host_at_home"],
}).sort_values("match_id").reset_index(drop=True)

d2 = pd.DataFrame({
    "match_id": teams["match_id"],
    "team": teams["team"],
    "opponent": teams["opponent"],
    "goals": teams["goals_for"],
    "team_fifa_points": teams["fifa_points"],
    "opp_fifa_points": teams["opp_fifa_points"],
    "team_gf_form": teams["gf_form"],
    "opp_ga_form": teams["opp_ga_form"],
    "rest_diff": teams["rest_diff"],
    "host_at_home": teams["host_at_home"],
    "knockout": teams["knockout"],
    "altitude": teams["altitude"],
}).sort_values(["match_id", "team"]).reset_index(drop=True)

features_1 = ["team1_fifa_points", "team2_fifa_points", "gf_form_diff", "ga_form_diff",
              "points_form_diff", "rest_diff", "travel_diff", "host_diff"]
features_2 = ["team_fifa_points", "opp_fifa_points", "team_gf_form", "opp_ga_form",
              "rest_diff", "host_at_home", "knockout", "altitude"]

print("Dataset 2.1:", d1.shape, "missing values:", d1.isna().sum().sum())
print("Dataset 2.2:", d2.shape, "missing values:", d2.isna().sum().sum())

d1.to_csv("data/dataset_2_1_goal_difference.csv", index=False)
d2.to_csv("data/dataset_2_2_team_goals.csv", index=False)

---
# Model 2.1: Goal difference

### Explore the data

In [ ]:
X1 = d1[features_1]
y1 = d1["goal_diff"]

print(d1[["goal_diff"] + features_1].describe().round(2))

fig, ax = plt.subplots(1, 2, figsize=(18, 6))
sns.countplot(x=y1, ax=ax[0])
ax[0].set_title("Goal difference")
sns.heatmap(d1[["goal_diff"] + features_1].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax[1])
ax[1].set_title("Correlation")
plt.tight_layout()
plt.show()

### Multicollinearity (VIF)

In [ ]:
X1_const = sm.add_constant(X1)
for i, col in enumerate(X1.columns):
    print(col, round(variance_inflation_factor(X1_const.values, i + 1), 2))

### Train / test split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X1, y1, test_size=0.2, random_state=2026)

model1 = LinearRegression()
model1.fit(X_train, y_train)

pred_train = model1.predict(X_train)
pred_test = model1.predict(X_test)
baseline = np.full(len(y_test), y_train.mean())

print("Train R2:", round(r2_score(y_train, pred_train), 3))
print("Test R2: ", round(r2_score(y_test, pred_test), 3))
print("Test MAE:", round(mean_absolute_error(y_test, pred_test), 3))
print("Test RMSE:", round(np.sqrt(mean_squared_error(y_test, pred_test)), 3))
print("Baseline test RMSE:", round(np.sqrt(mean_squared_error(y_test, baseline)), 3))

### 10-fold cross-validation
One of the sets of tests are small (only 104 matches) so we also perform 10-fold cross-validation.
We compare the full model with a baseline (predict the average) and with a model using only the two
ranking variables.

In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=2026)

def cv_scores(model, X, y, cv, groups=None):
    pred = cross_val_predict(model, X, y, cv=cv, groups=groups)
    return [round(mean_absolute_error(y, pred), 3),
            round(np.sqrt(mean_squared_error(y, pred)), 3),
            round(r2_score(y, pred), 3)]

cv1 = pd.DataFrame({
    "Baseline (mean)": cv_scores(DummyRegressor(), X1, y1, kf),
    "Ranking only": cv_scores(LinearRegression(), X1[["team1_fifa_points", "team2_fifa_points"]], y1, kf),
    "Full model (8 variables)": cv_scores(LinearRegression(), X1, y1, kf),
}, index=["MAE", "RMSE", "R2"]).T
cv1

### Coefficients and p-values

In [ ]:
ols1 = sm.OLS(y1, sm.add_constant(X1)).fit()
print(ols1.summary())

### Check assumptions

In [ ]:
resid1 = ols1.resid
fitted1 = ols1.fittedvalues

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(fitted1, resid1, alpha=0.6)
ax[0].axhline(0, color="red")
ax[0].set_xlabel("Fitted")
ax[0].set_ylabel("Residual")
ax[0].set_title("Residuals vs fitted")
sm.qqplot(resid1, line="45", fit=True, ax=ax[1])
ax[1].set_title("Q-Q plot")
plt.tight_layout()
plt.show()

print("Shapiro-Wilk p:", round(stats.shapiro(resid1).pvalue, 4))
print("Breusch-Pagan p:", round(het_breuschpagan(resid1, ols1.model.exog)[1], 4))
print("Durbin-Watson:", round(durbin_watson(resid1), 2))

### Results for Model 2.1
- The model explains about **31% of the variation** in goal difference on unseen matches (CV R2 = 0.305).
  Its error (RMSE 1.67) is lower than the baseline (2.02).
- Only the **FIFA ranking points** are significant (p < 0.001). 100 extra points for team1 add about
  0.47 goals to its goal difference, and 100 extra points for team2 take away about 0.65 goals.
- The form, rest, travel and home variables are not significant.
- The ranking-only model does slightly better (CV R2 = 0.358), so the extra six variables mostly add noise.
- Assumptions are OK: residuals look normal (Shapiro p = 0.33), variance is roughly constant
  (Breusch-Pagan p = 0.06) and Durbin-Watson is 2.1. All VIFs are below 5. The "large condition number"
  note in the summary is only because ranking points are big numbers (around 1,600).

---
# Model 2.2: Goals scored by a team

### Explore the data

In [ ]:
X2 = d2[features_2]
y2 = d2["goals"]
groups = d2["match_id"]

print(d2[["goals"] + features_2].describe().round(2))

fig, ax = plt.subplots(1, 2, figsize=(18, 6))
sns.countplot(x=y2, ax=ax[0])
ax[0].set_title("Goals scored")
sns.heatmap(d2[["goals"] + features_2].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax[1])
ax[1].set_title("Correlation")
plt.tight_layout()
plt.show()

### Multicollinearity (VIF)

In [ ]:
X2_const = sm.add_constant(X2)
for i, col in enumerate(X2.columns):
    print(col, round(variance_inflation_factor(X2_const.values, i + 1), 2))

### Train / test split (80/20)
There are two rows in each match (one for each team), so we split by match. Both teams of a match take the same.
set: if not set, else model would see half match while training.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=2026)
train_idx, test_idx = next(gss.split(X2, y2, groups))
X_train, X_test = X2.iloc[train_idx], X2.iloc[test_idx]
y_train, y_test = y2.iloc[train_idx], y2.iloc[test_idx]

model2 = LinearRegression()
model2.fit(X_train, y_train)

pred_train = model2.predict(X_train)
pred_test = model2.predict(X_test)
baseline = np.full(len(y_test), y_train.mean())

print("Train R2:", round(r2_score(y_train, pred_train), 3))
print("Test R2: ", round(r2_score(y_test, pred_test), 3))
print("Test MAE:", round(mean_absolute_error(y_test, pred_test), 3))
print("Test RMSE:", round(np.sqrt(mean_squared_error(y_test, pred_test)), 3))
print("Baseline test RMSE:", round(np.sqrt(mean_squared_error(y_test, baseline)), 3))

### 10-fold cross-validation (grouped by match)

In [ ]:
gkf = GroupKFold(n_splits=10)

cv2 = pd.DataFrame({
    "Baseline (mean)": cv_scores(DummyRegressor(), X2, y2, gkf, groups),
    "Ranking only": cv_scores(LinearRegression(), X2[["team_fifa_points", "opp_fifa_points"]], y2, gkf, groups),
    "Full model (8 variables)": cv_scores(LinearRegression(), X2, y2, gkf, groups),
}, index=["MAE", "RMSE", "R2"]).T
cv2

### Coefficients and p-values
Standard errors are clustered by match because the two rows of the same match are not independent.

In [ ]:
ols2 = sm.OLS(y2, sm.add_constant(X2)).fit(cov_type="cluster", cov_kwds={"groups": groups})
print(ols2.summary())

### Check assumptions

In [ ]:
resid2 = ols2.resid
fitted2 = ols2.fittedvalues

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(fitted2, resid2, alpha=0.6)
ax[0].axhline(0, color="red")
ax[0].set_xlabel("Fitted")
ax[0].set_ylabel("Residual")
ax[0].set_title("Residuals vs fitted")
sm.qqplot(resid2, line="45", fit=True, ax=ax[1])
ax[1].set_title("Q-Q plot")
plt.tight_layout()
plt.show()

print("Shapiro-Wilk p:", round(stats.shapiro(resid2).pvalue, 4))
print("Breusch-Pagan p:", round(het_breuschpagan(resid2, ols2.model.exog)[1], 4))

### Results for Model 2.2
- The model accounts for approximately 18% of the variation in goals scored on an unseen match (CV R2 = 0.176).
  It has a smaller error (RMSE 1.26) than the baseline (1.39).
- **Team ranking points** (p < 0.001) and **opponent ranking points** (p = 0.007) are significant.
  100 extra points add about 0.34 goals; 100 extra points for the opponent take away about 0.19 goals.
- Host nations playing at home scored about 0.75 more goals, but this is just above 0.05 (p = 0.067).
- The ranking-only model again does about as well (CV R2 = 0.194).
- Assumptions are not fully met: residuals are skewed (Shapiro p < 0.001) and variance is not constant
  (Breusch-Pagan p = 0.018). This is because goals are counts (0, 1, 2...). A Poisson regression
  would suit this data better.

---
# Summary

In [ ]:
summary = pd.DataFrame({
    "Model 2.1": [len(d1), cv1.loc["Full model (8 variables)", "R2"], cv1.loc["Full model (8 variables)", "RMSE"],
                  cv1.loc["Baseline (mean)", "RMSE"], round(ols1.rsquared_adj, 3)],
    "Model 2.2": [len(d2), cv2.loc["Full model (8 variables)", "R2"], cv2.loc["Full model (8 variables)", "RMSE"],
                  cv2.loc["Baseline (mean)", "RMSE"], round(ols2.rsquared_adj, 3)],
}, index=["Rows", "CV R2", "CV RMSE", "Baseline RMSE", "Adjusted R2"])
summary.to_csv("data/summary_regression.csv")
summary

## Conclusion
- Both models predict better than just guessing the average.
- In both models, FIFA ranking points are the best predictor.
- Model 2.1 is more efficient than Model 2.2, because the number of goals in one team is a small random number.
- Form, schedule variables could be useful with more matches, but are not with 104 matches.